# ChurnSight — Exploratory Spark Notebook

Step-by-step version of the batch pipeline: load, clean and enrich the telecom and audit datasets, run Spark SQL KPIs, and write Parquet.
Run it from the project folder (paths are relative).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when, round

spark = SparkSession.builder \
    .appName("ChurnSight") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/08 06:49:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
telco_df = spark.read.csv(
    "Telco.csv",
    header=True,
    inferSchema=True
)

risk_df = spark.read.csv(
    "risk.csv",
    header=True,
    inferSchema=True
)

In [3]:
telco_df.printSchema()
telco_df.show(5)

risk_df.printSchema()
risk_df.show(5)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+
|Year|Firm_Name|Total_Audit_Engagements|High_Risk_Cases|Compliance_Violations|Fraud_Cases_Detected|Industry_Affected|Total_Revenue_Impact|AI_Used_for_Auditing|Employee_Workload|Audit_Effectiveness_Score|Client_Satisfaction_Score|
+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+
|2020|      PwC|                   2829|             51|                  123|                  39|       Healthcare|              114.24|                  No|               57|                      5.8|                      8.4|
|2022| Deloitte|                   3589|            185|                   30|  

In [4]:
telco_df = telco_df.dropDuplicates()

In [5]:
risk_df = risk_df.dropDuplicates()

In [6]:
for field in telco_df.schema.fields:
    if isinstance(field.dataType, StringType):
        telco_df = telco_df.withColumn(field.name, trim(col(field.name)))

In [7]:
telco_df = telco_df.withColumn(
    "TotalCharges",
    when(col("TotalCharges") == "", None)
    .otherwise(col("TotalCharges").cast("double"))
)

telco_df = telco_df.fillna({"TotalCharges": 0.0})

In [8]:
binary_cols_to_clean = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for c in binary_cols_to_clean:
    telco_df = telco_df.withColumn(
        c,
        when(
            col(c).isin("No internet service", "No phone service"),
            "No"
        ).otherwise(col(c))
    )

In [9]:
telco_df = telco_df.withColumn(
    "AvgMonthlySpend",
    round(
        col("TotalCharges") /
        when(col("tenure") == 0, 1).otherwise(col("tenure")),
        2
    )
)

telco_df = telco_df.withColumn(
    "TenureYears",
    round(col("tenure") / 12, 1)
)

telco_df = telco_df.withColumn(
    "TenureGroup",
    when(col("tenure") <= 12, "0-1 Year")
    .when(col("tenure") <= 24, "1-2 Years")
    .when(col("tenure") <= 48, "2-4 Years")
    .otherwise("4+ Years")
)


# Numeric churn flag (1 = left, 0 = stayed) so churn rate = AVG(ChurnFlag)
telco_df = telco_df.withColumn(
    "ChurnFlag",
    when(col("Churn") == "Yes", 1).otherwise(0).cast("double")
)

In [10]:
telco_df.cache()

DataFrame[customerID: string, gender: string, SeniorCitizen: int, Partner: string, Dependents: string, tenure: int, PhoneService: string, MultipleLines: string, InternetService: string, OnlineSecurity: string, OnlineBackup: string, DeviceProtection: string, TechSupport: string, StreamingTV: string, StreamingMovies: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, MonthlyCharges: double, TotalCharges: double, Churn: string, AvgMonthlySpend: double, TenureYears: double, TenureGroup: string, ChurnFlag: double]

In [11]:
telco_df.show(5)
telco_df.printSchema()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+---------------+-----------+-----------+---------+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|AvgMonthlySpend|TenureYears|TenureGroup|ChurnFlag|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+---------------+-----------+-----------+---------+
|8637-XJIVR|Female|      

In [12]:
for field in risk_df.schema.fields:
    if isinstance(field.dataType, StringType):
        risk_df = risk_df.withColumn(field.name, trim(col(field.name)))

In [13]:
risk_df = risk_df.fillna({
    "High_Risk_Cases": 0,
    "Fraud_Cases_Detected": 0,
    "Total_Revenue_Impact": 0.0,
    "Audit_Effectiveness_Score": 0.0,
    "Client_Satisfaction_Score": 0.0
})

In [14]:
risk_df = risk_df.withColumn(
    "High_Risk_Ratio",
    round(
        col("High_Risk_Cases") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        4
    )
)

In [15]:
risk_df = risk_df.withColumn(
    "Fraud_Detection_Rate",
    round(
        col("Fraud_Cases_Detected") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        4
    )
)

In [16]:
risk_df = risk_df.withColumn(
    "Avg_Revenue_Impact_Per_Engagement",
    round(
        col("Total_Revenue_Impact") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        2
    )
)

In [17]:
risk_df = risk_df.withColumn(
    "Audit_Quality_Score",
    round(
        (col("Audit_Effectiveness_Score") +
         col("Client_Satisfaction_Score")) / 2,
        2
    )
)

In [18]:
risk_df.cache()


DataFrame[Year: int, Firm_Name: string, Total_Audit_Engagements: int, High_Risk_Cases: int, Compliance_Violations: int, Fraud_Cases_Detected: int, Industry_Affected: string, Total_Revenue_Impact: double, AI_Used_for_Auditing: string, Employee_Workload: int, Audit_Effectiveness_Score: double, Client_Satisfaction_Score: double, High_Risk_Ratio: double, Fraud_Detection_Rate: double, Avg_Revenue_Impact_Per_Engagement: double, Audit_Quality_Score: double]

In [19]:
risk_df.show(5)
risk_df.printSchema()

+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+---------------+--------------------+---------------------------------+-------------------+
|Year|Firm_Name|Total_Audit_Engagements|High_Risk_Cases|Compliance_Violations|Fraud_Cases_Detected|Industry_Affected|Total_Revenue_Impact|AI_Used_for_Auditing|Employee_Workload|Audit_Effectiveness_Score|Client_Satisfaction_Score|High_Risk_Ratio|Fraud_Detection_Rate|Avg_Revenue_Impact_Per_Engagement|Audit_Quality_Score|
+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+---------------+--------------------+---------------------------------+-------------------+
|2021|     KPMG|                   25

In [20]:
telco_df.createOrReplaceTempView("telco")

In [21]:
#Spark SQL
spark.sql("""
SELECT COUNT(*) AS Total_Customers
FROM telco
""").show()

+---------------+
|Total_Customers|
+---------------+
|           7043|
+---------------+



In [22]:
spark.sql("""
SELECT ROUND(AVG(ChurnFlag) * 100, 2) AS Churn_Rate
FROM telco
""").show()

+----------+
|Churn_Rate|
+----------+
|     26.54|
+----------+



In [23]:
spark.sql("""
SELECT Contract,ROUND(SUM(TotalCharges),2) AS Revenue
FROM telco
GROUP BY Contract
ORDER BY Revenue DESC
""").show()

+--------------+---------+
|      Contract|  Revenue|
+--------------+---------+
|      Two year|6283253.7|
|Month-to-month|5305861.5|
|      One year|4467053.5|
+--------------+---------+



In [24]:
spark.sql("""
SELECT TenureGroup,ROUND(AVG(AvgMonthlySpend),2) AS Avg_Spend
FROM telco
GROUP BY TenureGroup
ORDER BY Avg_Spend DESC
""").show()

+-----------+---------+
|TenureGroup|Avg_Spend|
+-----------+---------+
|   4+ Years|    73.97|
|  2-4 Years|    65.87|
|  1-2 Years|     61.3|
|   0-1 Year|    55.94|
+-----------+---------+



In [25]:
risk_df.createOrReplaceTempView("financial")

In [26]:
spark.sql("""
SELECT ROUND(AVG(High_Risk_Ratio),4) AS Avg_Risk
FROM financial
""").show()

+--------+
|Avg_Risk|
+--------+
|   0.134|
+--------+



In [27]:
spark.sql("""
SELECT Industry_Affected,ROUND(AVG(High_Risk_Ratio),4) AS Risk
FROM financial
GROUP BY Industry_Affected
ORDER BY Risk DESC
""").show()

+-----------------+------+
|Industry_Affected|  Risk|
+-----------------+------+
|          Finance|0.1534|
|       Healthcare|0.1438|
|             Tech| 0.126|
|           Retail|0.1194|
+-----------------+------+



In [28]:
spark.sql("""
SELECT AI_Used_for_Auditing,ROUND(AVG(Audit_Quality_Score),2) AS Avg_Quality
FROM financial
GROUP BY AI_Used_for_Auditing
""").show()

+--------------------+-----------+
|AI_Used_for_Auditing|Avg_Quality|
+--------------------+-----------+
|                  No|       7.36|
|                 Yes|       7.48|
+--------------------+-----------+



In [29]:
spark.sql("""
SELECT Firm_Name,ROUND(SUM(Total_Revenue_Impact),2) AS Revenue_Impact
FROM financial
GROUP BY Firm_Name
ORDER BY Revenue_Impact DESC
LIMIT 10
""").show()

+-------------+--------------+
|    Firm_Name|Revenue_Impact|
+-------------+--------------+
|     Deloitte|       8547.44|
|Ernst & Young|       6862.86|
|          PwC|       6108.49|
|         KPMG|       5735.11|
+-------------+--------------+



In [30]:
telco_df.write.mode("overwrite").parquet("output/telco")

26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/10/08 06:49:18 WARN MemoryManager: Total allocation exceeds 95.

26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 42.22% for 18 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 44.71% for 17 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 9

26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/08 06:49:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [31]:
risk_df.write.mode("overwrite").parquet("output/risk")